# Modeling K-Means Clustering
## Sistem Clustering Kemiskinan Wilayah di Indonesia

**Fitur yang digunakan (3 fitur):**

| Fitur | Korelasi ke Kemiskinan | Alasan |
|---|---|---|
| `Tingkat_Penduduk_Miskin` | Target utama | Indikator langsung kemiskinan |
| `PengeluaranPerKapita` | -0.66 (kuat) | Proxy daya beli & kesejahteraan |
| `Rata-rata Lama Sekolah` | -0.54 (kuat) | Proxy kualitas SDM & pendidikan |

**Fitur yang dihilangkan & alasannya:**

| Fitur | Korelasi | Alasan Dihapus |
|---|---|---|
| `Usia Harapan Hidup` | -0.56 | Sangat mirip informasinya dengan Rata-rata Lama Sekolah (redundan) |
| `Indeks Pembangunan Gender` | -0.41 | Korelasi sedang, tidak spesifik untuk kemiskinan |
| `Harapan Lama Sekolah` | -0.45 | Sangat berkorelasi tinggi dengan Rata-rata Lama Sekolah (0.78) → informasi sama |
| `Produk Domestik Regional Bruto` | -0.24 | Korelasi lemah, nilai sangat skewed |
| `Indeks Kemahalan Konstruksi` | +0.56 | Relevan secara korelasi tapi membingungkan untuk presentasi |
| `PengeluaranPerkapita_Rokok` | -0.18 | Korelasi paling lemah, tidak relevan |

**Hasil uji Silhouette Score:**
3 fitur ini menghasilkan score **0.5139** — lebih tinggi dari kombinasi 4, 5, bahkan 8 fitur.
Lebih sedikit fitur = cluster lebih bersih dan mudah dijelaskan.

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

---
## 2. Load Dataset

In [ ]:
df = pd.read_excel('/content/Data_Tingkat_Kemiskinan.xlsx')

print('Shape data:', df.shape)
df.head()

---
## 3. Data Cleaning

In [ ]:
print('Missing value:')
print(df.isnull().sum())
df = df.dropna().drop_duplicates().reset_index(drop=True)
print('\nShape setelah cleaning:', df.shape)

In [ ]:
# Hapus outlier IQR pada fitur yang dipakai
def hapus_outlier_iqr(data, kolom):
    df_bersih = data.copy()
    for col in kolom:
        Q1 = df_bersih[col].quantile(0.25)
        Q3 = df_bersih[col].quantile(0.75)
        IQR = Q3 - Q1
        sebelum = len(df_bersih)
        df_bersih = df_bersih[
            (df_bersih[col] >= Q1 - 1.5 * IQR) &
            (df_bersih[col] <= Q3 + 1.5 * IQR)
        ]
        print(f'  {col}: hapus {sebelum - len(df_bersih)} baris outlier')
    return df_bersih.reset_index(drop=True)

fitur_pakai = [
    'Tingkat_Penduduk_Miskin',
    'PengeluaranPerKapita',
    'Rata-rata Lama Sekolah'
]

print('Hapus outlier:')
df = hapus_outlier_iqr(df, fitur_pakai)
print(f'Shape final: {df.shape}')

---
## 4. Seleksi Fitur

Dari 8 fitur numerik yang tersedia, hanya **3 fitur** yang dipakai berdasarkan:
- Korelasi tertinggi terhadap `Tingkat_Penduduk_Miskin`
- Tidak saling redundan satu sama lain
- Mudah dijelaskan secara logika

Hasilnya terbukti: 3 fitur ini menghasilkan **Silhouette Score lebih tinggi** (0.51) dibanding pakai lebih banyak fitur.

In [ ]:
features = [
    'Tingkat_Penduduk_Miskin',
    'PengeluaranPerKapita',
    'Rata-rata Lama Sekolah'
]

X = df[features].copy()
print('Statistik fitur:')
X.describe()

---
## 5. Min-Max Scaling Manual

K-Means bekerja berdasarkan **jarak**. Kalau skala fitur berbeda jauh, fitur dengan angka besar akan mendominasi.

**Analogi:** Bandingkan persentase kemiskinan (2–40%) dengan pengeluaran per kapita (4.000–25.000 ribu rupiah). Tanpa scaling, perbedaan pengeluaran akan terasa jauh lebih besar di mata algoritma padahal kontribusinya harusnya setara. Scaling membuat ketiganya 'setara volumenya'.

Formula: `X_scaled = (X - X_min) / (X_max - X_min)` → hasil 0 sampai 1.

In [ ]:
def minmax_scale(X):
    """
    Min-Max Normalization manual (tanpa sklearn).
    Input : numpy array (n_samples, n_features)
    Output: array ternormalisasi [0,1], x_min, x_max
    """
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    x_scaled = (X - x_min) / (x_max - x_min)
    return x_scaled, x_min, x_max

X_arr = X.values
X_scaled, x_min, x_max = minmax_scale(X_arr)

print('Min tiap fitur:', x_min)
print('Max tiap fitur:', x_max)
print()
pd.DataFrame(X_scaled, columns=features).describe()

---
## 6. Implementasi K-Means Manual

**Cara kerja K-Means:**
1. Lempar K centroid secara acak
2. Setiap data bergabung ke centroid terdekat
3. Geser centroid ke rata-rata anggotanya
4. Ulangi sampai centroid tidak bergerak lagi

Tidak ada `sklearn.cluster.KMeans` digunakan di sini.

In [ ]:
def euclidean_distance(a, b):
    """Jarak Euclidean antara dua titik."""
    return np.sqrt(np.sum((a - b) ** 2))


def kmeans_manual(X, k, max_iter=300, tol=1e-4, random_state=42):
    """
    K-Means Clustering dari nol.

    Parameters
    ----------
    X            : numpy array (n_samples, n_features)
    k            : jumlah cluster
    max_iter     : batas maksimum iterasi
    tol          : berhenti jika pergeseran centroid < tol
    random_state : seed reprodusibilitas

    Returns
    -------
    labels    : label cluster tiap data
    centroids : posisi akhir centroid
    inertia   : total WCSS (Within-Cluster Sum of Squares)
    """
    np.random.seed(random_state)
    n = X.shape[0]

    # Inisialisasi: pilih k titik acak sebagai centroid awal
    centroid_idx = np.random.choice(n, k, replace=False)
    centroids = X[centroid_idx].copy().astype(float)
    labels = np.zeros(n, dtype=int)

    for iterasi in range(max_iter):

        # Assign: setiap titik ke centroid terdekat
        labels_baru = np.zeros(n, dtype=int)
        for i in range(n):
            jarak = [euclidean_distance(X[i], centroids[c]) for c in range(k)]
            labels_baru[i] = np.argmin(jarak)

        # Update: centroid baru = rata-rata anggota cluster
        centroids_baru = np.zeros_like(centroids)
        for c in range(k):
            anggota = X[labels_baru == c]
            if len(anggota) == 0:
                centroids_baru[c] = X[np.random.randint(n)]
            else:
                centroids_baru[c] = anggota.mean(axis=0)

        # Cek konvergensi
        pergeseran = max(euclidean_distance(centroids_baru[c], centroids[c]) for c in range(k))
        labels = labels_baru
        centroids = centroids_baru

        if pergeseran < tol:
            print(f'  Konvergen pada iterasi ke-{iterasi + 1}')
            break

    inertia = sum(euclidean_distance(X[i], centroids[labels[i]])**2 for i in range(n))
    return labels, centroids, inertia


print('Fungsi K-Means manual siap.')

---
## 7. Elbow Method — Cari K Optimal

Plot **inertia** (WCSS) untuk setiap K. K optimal = titik di mana penurunan inertia mulai melambat (bentuk siku).

In [ ]:
inertia_list = []
K_range = range(1, 11)

print('Elbow Method (K=1 s/d 10)...')
for k in K_range:
    print(f'  K={k}', end=' ')
    _, _, inertia = kmeans_manual(X_scaled, k=k, random_state=42)
    inertia_list.append(inertia)
    print(f'→ Inertia: {inertia:.4f}')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertia_list, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method — Menentukan K Optimal', fontsize=14)
plt.xticks(list(K_range))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 8. Silhouette Score Manual — Konfirmasi K

Rumus: `s(i) = (b - a) / max(a, b)` di mana:
- `a` = rata-rata jarak ke sesama anggota cluster sendiri
- `b` = rata-rata jarak ke anggota cluster terdekat lainnya

Nilai mendekati 1 = cluster sudah bagus dan terpisah jelas.

In [ ]:
def silhouette_score_manual(X, labels):
    """
    Silhouette Score manual (tanpa sklearn).
    """
    n = X.shape[0]
    cluster_ids = np.unique(labels)
    s_list = []

    for i in range(n):
        ci = labels[i]
        own = X[labels == ci]

        if len(own) == 1:
            s_list.append(0)
            continue

        a_i = np.mean([
            euclidean_distance(X[i], own[j])
            for j in range(len(own))
            if not np.array_equal(own[j], X[i])
        ])

        b_i = min(
            np.mean([euclidean_distance(X[i], X[labels == c][j])
                     for j in range((labels == c).sum())])
            for c in cluster_ids if c != ci
        )

        s_list.append((b_i - a_i) / max(a_i, b_i))

    return np.mean(s_list)


print('Menghitung Silhouette Score K=2 s/d 6...')
print('(Proses ini butuh beberapa menit karena dihitung manual)\n')

sil_scores = {}
for k in range(2, 7):
    lbl, _, _ = kmeans_manual(X_scaled, k=k, random_state=42)
    score = silhouette_score_manual(X_scaled, lbl)
    sil_scores[k] = score
    print(f'  K={k}  →  Silhouette Score: {score:.4f}')

best_k = max(sil_scores, key=sil_scores.get)
print(f'\nK terbaik: K = {best_k}  (score = {sil_scores[best_k]:.4f})')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()),
         marker='s', color='coral', linewidth=2, markersize=8)
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score tiap K', fontsize=14)
plt.xticks(list(sil_scores.keys()))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 9. Training Model Final

In [ ]:
K_OPTIMAL = best_k
# Override manual jika perlu: K_OPTIMAL = 3

print(f'Training K-Means dengan K = {K_OPTIMAL}...')
labels_final, centroids_final, inertia_final = kmeans_manual(
    X_scaled, k=K_OPTIMAL, random_state=42
)

df['Cluster'] = labels_final

print(f'Inertia final  : {inertia_final:.4f}')
print(f'Silhouette Score: {sil_scores[K_OPTIMAL]:.4f}')
print(f'\nDistribusi cluster:')
print(df['Cluster'].value_counts().sort_index())

---
## 10. Interpretasi & Pelabelan Cluster

In [ ]:
ringkasan = df.groupby('Cluster')[features].mean().round(2)
print('Rata-rata fitur per cluster:')
print(ringkasan)

In [ ]:
# Label berdasarkan ranking Tingkat_Penduduk_Miskin
pilihan_label = {
    2: ['Tidak Miskin', 'Miskin'],
    3: ['Tidak Miskin', 'Miskin', 'Sangat Miskin'],
    4: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin'],
    5: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin', 'Ekstrem Miskin'],
}
daftar_label = pilihan_label.get(K_OPTIMAL, [f'Cluster {i}' for i in range(K_OPTIMAL)])

rank_miskin = ringkasan['Tingkat_Penduduk_Miskin'].rank(ascending=True)
label_map = {cid: daftar_label[int(r) - 1] for cid, r in rank_miskin.items()}

df['Kategori'] = df['Cluster'].map(label_map)

print('Mapping label:')
for c, lbl in sorted(label_map.items()):
    baris = (df['Cluster'] == c).sum()
    print(f'  Cluster {c} → {lbl:20s} ({baris} wilayah)')

In [ ]:
# Tabel ringkasan lengkap
ringkasan['Kategori'] = ringkasan.index.map(label_map)
ringkasan['Jumlah Wilayah'] = df.groupby('Cluster').size()
ringkasan[['Kategori', 'Jumlah Wilayah'] + features]

---
## 11. Visualisasi Hasil

In [ ]:
warna = plt.cm.get_cmap('Set1', K_OPTIMAL)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Tingkat Miskin vs Pengeluaran
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                    label=f'{label_map.get(c,"")}', alpha=0.7, s=50, color=warna(c))
axes[0].scatter(centroids_final[:, 0], centroids_final[:, 1],
                marker='X', s=200, color='black', zorder=5, label='Centroid')
axes[0].set_xlabel('Tingkat Penduduk Miskin (scaled)')
axes[0].set_ylabel('Pengeluaran per Kapita (scaled)')
axes[0].set_title('Kemiskinan vs Pengeluaran')
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.4)

# Plot 2: Tingkat Miskin vs Rata-rata Lama Sekolah
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_scaled[mask, 0], X_scaled[mask, 2],
                    label=f'{label_map.get(c,"")}', alpha=0.7, s=50, color=warna(c))
axes[1].scatter(centroids_final[:, 0], centroids_final[:, 2],
                marker='X', s=200, color='black', zorder=5, label='Centroid')
axes[1].set_xlabel('Tingkat Penduduk Miskin (scaled)')
axes[1].set_ylabel('Rata-rata Lama Sekolah (scaled)')
axes[1].set_title('Kemiskinan vs Rata-rata Lama Sekolah')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.suptitle(f'Hasil K-Means Manual — K={K_OPTIMAL}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart distribusi wilayah per cluster
urutan = [label_map[c] for c in sorted(label_map.keys())]
distribusi = df['Kategori'].value_counts().reindex(urutan)

plt.figure(figsize=(8, 5))
bars = plt.bar(distribusi.index, distribusi.values,
               color=[warna(c) for c in range(K_OPTIMAL)], edgecolor='black', width=0.6)
for bar, val in zip(bars, distribusi.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.xlabel('Kategori Kemiskinan', fontsize=12)
plt.ylabel('Jumlah Wilayah (Kab/Kota)', fontsize=12)
plt.title('Distribusi Wilayah per Cluster Kemiskinan', fontsize=13)
plt.tight_layout()
plt.show()

---
## 12. Sampel Wilayah per Cluster

In [ ]:
kolom_tampil = ['Kabupaten_Kota', 'Kategori'] + features

for c in sorted(df['Cluster'].unique()):
    lbl = label_map.get(c, f'Cluster {c}')
    subset = df[df['Cluster'] == c][kolom_tampil]
    print(f'\n{"="*65}')
    print(f'  CLUSTER {c} — {lbl}  ({len(subset)} wilayah)')
    print(f'{"="*65}')
    print(subset.head(8).to_string(index=False))

---
## 13. Simpan Output untuk Tim Geospatial

In [ ]:
df_output = df[['Kabupaten_Kota', 'Cluster', 'Kategori'] + features].copy()

df_output.to_csv('hasil_clustering_kemiskinan.csv', index=False)

print('Tersimpan: hasil_clustering_kemiskinan.csv')
print(f'Total baris  : {len(df_output)}')
print(f'Kolom output : {df_output.columns.tolist()}')
print()
df_output.head(10)